In [1]:
import numpy as np
import librosa
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 1. Feature Extraction Function (MFCC extraction)
def extract_audio_features(file_path, max_pad_len=174):
    try:
        # Load audio at 22050 Hz sampling rate
        audio, sample_rate = librosa.load(file_path, sr=22050, res_type='kaiser_fast')
        mfccs = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)

        # Pad or truncate length for uniform input dimension
        if mfccs.shape[1] < max_pad_len:
            pad_width = max_pad_len - mfccs.shape[1]
            mfccs = np.pad(mfccs, pad_width=((0, 0), (0, pad_width)), mode='constant')
        else:
            mfccs = mfccs[:, :max_pad_len]
        return mfccs
    except Exception as e:
        return None

# 2. Generating Synthetic Matrix Representation for Validation
# (Replace this block with directory iteration for RAVDESS or TESS dataset files)
num_samples = 600
n_mfcc = 40
time_steps = 174
num_classes = 4  # e.g., Angry, Happy, Sad, Neutral

X_features = np.random.randn(num_samples, n_mfcc, time_steps)
labels = np.random.choice(['angry', 'happy', 'sad', 'neutral'], size=num_samples)

le = LabelEncoder()
y_encoded = tf.keras.utils.to_categorical(le.fit_transform(labels))

# Reshape input to (samples, time_steps, features) for Conv1D
X_features = np.transpose(X_features, (0, 2, 1))

X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_encoded, test_size=0.2, random_state=42
)

# 3. 1D CNN Architecture
model = models.Sequential([
    layers.Conv1D(64, kernel_size=5, activation='relu', input_shape=(time_steps, n_mfcc)),
    layers.BatchNormalization(),
    layers.MaxPooling1D(pool_size=2),
    layers.Dropout(0.3),

    layers.Conv1D(128, kernel_size=5, activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling1D(pool_size=2),
    layers.Dropout(0.3),

    layers.GlobalAveragePooling1D(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 4. Training
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=15,
    batch_size=32,
    verbose=1
)

# 5. Model Evaluation
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nFinal Test Loss: {test_loss:.4f} | Final Test Accuracy: {test_acc:.4f}")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/15
15/15 ━━━━━━━━━━━━━━━━━━━━ 4s 66ms/step - accuracy: 0.2521 - loss: 1.4623 - val_accuracy: 0.2667 - val_loss: 1.6108
Epoch 2/15
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - accuracy: 0.3021 - loss: 1.3700 - val_accuracy: 0.2167 - val_loss: 1.4602
Epoch 3/15
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step - accuracy: 0.3479 - loss: 1.3477 - val_accuracy: 0.1917 - val_loss: 1.4353
Epoch 4/15
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - accuracy: 0.3688 - loss: 1.3150 - val_accuracy: 0.2667 - val_loss: 1.4273
Epoch 5/15
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - accuracy: 0.4146 - loss: 1.2666 - val_accuracy: 0.2583 - val_loss: 1.4922
Epoch 6/15
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step - accuracy: 0.5021 - loss: 1.2086 - val_accuracy: 0.2583 - val_loss: 1.4811
Epoch 7/15
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 44ms/step - accuracy: 0.4917 - loss: 1.1617 - val_accuracy: 0.2583 - val_loss: 1.4836
Epoch 8/15
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 44ms/step - accuracy: 0.5833 - loss: 1.0962 - val_accuracy: 0.2583 - v